# 07 — Feature Group Optimisation


In [ ]:
import itertools
import json
import sys
from collections import defaultdict
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pymetis
import seaborn as sns
from matplotlib.patches import Patch
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
from sklearn.cluster import KMeans

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.partitioning import simulated_annealing_partition
from build_optimiser.simulation import rebuild_cost, simulate_merge, simulate_split

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Load Feature Groups

Read feature group assignments, thin dependency analysis, and target ownership from NB05/NB06 outputs.


In [ ]:
# ── Load feature group assignments (from NB06) ────────────────────────────────
fg_df = pd.read_parquet('../data/processed/feature_group_assignments.parquet')
group_map = dict(zip(fg_df['cmake_target'], fg_df['feature_group']))

# ── Load thin dependencies (from NB05) ────────────────────────────────────────
thin_dep_path = Path('../data/results/thin_dependencies.csv')
if thin_dep_path.exists():
    thin_df = pd.read_csv(thin_dep_path)
else:
    thin_df = pd.DataFrame(columns=['depending_target', 'depended_target',
                                    'used_headers', 'total_headers', 'thinness_ratio'])
    print('Warning: thin_dependencies.csv not found. Run NB05 first.')

thin_set = set(
    zip(thin_df['depending_target'], thin_df['depended_target'])
) if not thin_df.empty else set()

# ── Load target ownership (from contributor analysis) ────────────────────────
ownership_path = Path('../data/processed/target_ownership.parquet')
groups_path = Path('../data/processed/contributor_groups.parquet')
if ownership_path.exists():
    ownership_df = pd.read_parquet(ownership_path)
    # Derive owning_group_label if not present
    if 'owning_group_label' not in ownership_df.columns and 'owning_group_id' in ownership_df.columns and groups_path.exists():
        groups_df = pd.read_parquet(groups_path)
        label_map = groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label']
        ownership_df['owning_group_label'] = ownership_df['owning_group_id'].map(label_map).fillna('unknown')
    ownership_map = dict(zip(ownership_df['cmake_target'], ownership_df.get('owning_group_label', ownership_df.get('owning_group_id', 'unknown'))))
else:
    ownership_df = pd.DataFrame()
    ownership_map = {}
    print('Warning: target_ownership.parquet not found.')

# ── Load executable-library matrix (from NB05) ───────────────────────────────
exe_lib_path = Path('../data/processed/exe_library_matrix.parquet')
if exe_lib_path.exists():
    exe_lib_df = pd.read_parquet(exe_lib_path)
else:
    from build_optimiser.features import compute_exe_library_matrix
    exe_lib_df = compute_exe_library_matrix(G, target_df[['cmake_target', 'target_type']])
    print('Recomputed exe-library matrix.')

# ── Summary ───────────────────────────────────────────────────────────────────
all_groups = sorted(fg_df['feature_group'].unique())
feature_groups_only = [g for g in all_groups if g != 'core']
core_targets = set(fg_df[fg_df['feature_group'] == 'core']['cmake_target'])

print(f'Targets in graph:       {G.number_of_nodes():>6}')
print(f'Feature groups (incl. core): {len(all_groups):>3}')
print(f'Non-core feature groups:{len(feature_groups_only):>3}')
print(f'Core targets:           {len(core_targets):>6}')
print(f'Thin dependencies:      {len(thin_df):>6}')
print()
group_sizes = fg_df.groupby('feature_group')['cmake_target'].count().sort_values(ascending=False)
print('Group sizes:')
display(group_sizes.to_frame('target_count').reset_index())

## 2. Cross-Group Dependency Analysis

Classify edges crossing feature group boundaries. Rank by how many executables are forced to enable an extra group because of each cross-group edge.


In [ ]:
# ── Helper: count executables forced into group B solely by edge A→B ─────────
exe_group_required: dict[str, set[str]] = defaultdict(set)
for _, row in exe_lib_df.iterrows():
    lib = row['library']
    grp = group_map.get(lib)
    if grp and grp != 'core':
        exe_group_required[row['executable']].add(grp)

# ── Enumerate cross-group edges ───────────────────────────────────────────────
cross_rows = []
for _, row in edge_df.iterrows():
    src, dst = row['source_target'], row['dest_target']
    grp_src = group_map.get(src)
    grp_dst = group_map.get(dst)
    if not grp_src or not grp_dst:
        continue
    if grp_src == grp_dst:
        continue
    if grp_dst == 'core':
        continue  # core deps are expected everywhere

    is_thin = (src, dst) in thin_set

    # Count executables that would be forced into grp_dst by this edge
    # Proxy: executables in grp_src's group that currently need grp_dst
    forced_exes = sum(
        1 for exe, grps in exe_group_required.items()
        if grp_dst in grps and group_map.get(exe) != grp_dst
    )

    dep_type = row.get('dependency_type', 'unknown')
    scope = row.get('scope', 'unknown')
    is_direct = bool(row.get('is_direct', True))

    cross_rows.append({
        'source_target': src,
        'dest_target': dst,
        'source_group': grp_src,
        'dest_group': grp_dst,
        'is_thin': is_thin,
        'is_direct': is_direct,
        'dependency_type': dep_type,
        'scope': scope,
        'forced_executables': forced_exes,
    })

if cross_rows:
    cross_df = pd.DataFrame(cross_rows).sort_values('forced_executables', ascending=False)
else:
    cross_df = pd.DataFrame(cross_rows)
    print('No cross df found (dataset too small).')

print(f'Total cross-group edges: {len(cross_df)}')
print(f'  Thin:   {cross_df["is_thin"].sum()}')
print(f'  Direct: {cross_df["is_direct"].sum()}')
print()

# ── Top cross-group edges ─────────────────────────────────────────────────────
print('Top 20 cross-group edges by executables forced:')
display(cross_df[['source_target', 'source_group', 'dest_target', 'dest_group',
                   'is_thin', 'scope', 'forced_executables']].head(20).reset_index(drop=True))

# ── Heatmap: cross-group edge counts ─────────────────────────────────────────
groups_order = sorted(all_groups)
heat_data = cross_df.groupby(['source_group', 'dest_group'])['source_target'].count().unstack(fill_value=0)
heat_data = heat_data.reindex(index=groups_order, columns=groups_order, fill_value=0)

fig, ax = plt.subplots(figsize=(max(8, len(groups_order)), max(6, len(groups_order) * 0.8)))
sns.heatmap(heat_data, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Cross-Group Edge Count Heatmap (source → dest)')
ax.set_xlabel('Destination group')
ax.set_ylabel('Source group')
plt.tight_layout()
plt.show()


## 3. Target Split Proposals

For high-impact cross-group edges involving large targets: build the file-level include graph, apply spectral partitioning (Fiedler vector) and METIS, enforce codegen co-location, evaluate result.


In [ ]:
# ── Identify split candidates: large targets on high-impact cross-group edges ─
def safe_int(val, default=0):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return default
    try:
        return int(val)
    except (ValueError, TypeError):
        return default


target_metrics_idx = target_df.set_index('cmake_target')

split_candidates_set = set()
for _, row in cross_df[cross_df['forced_executables'] > 0].iterrows():
    for t in [row['source_target'], row['dest_target']]:
        if t not in target_metrics_idx.index:
            continue
        mr = target_metrics_idx.loc[t]
        if safe_int(mr.get('authored_file_count')) >= 4:
            split_candidates_set.add(t)

# Also include large targets even without forced_executables
EXCLUDE_TYPES = {'INTERFACE_LIBRARY', 'UTILITY'}
for t, mr in target_metrics_idx.iterrows():
    if mr.get('target_type') in EXCLUDE_TYPES or '::' in str(t):
        continue
    if t not in G:
        continue
    grp = group_map.get(t)
    if grp and grp != 'core' and safe_int(mr.get('authored_file_count')) >= 8:
        split_candidates_set.add(t)

print(f'Split candidates: {len(split_candidates_set)}')


# ── File-level include graph builder ─────────────────────────────────────────
def parse_header_tree(json_str):
    if json_str is None or (isinstance(json_str, float) and pd.isna(json_str)):
        return []
    try:
        return [(int(d), str(p)) for d, p in json.loads(json_str)]
    except (json.JSONDecodeError, ValueError, TypeError):
        return []


def build_intra_target_graph(target_name):
    target_files = file_df[file_df['cmake_target'] == target_name].copy()
    if target_files.empty:
        return nx.DiGraph()
    source_paths = set(target_files['source_file'].dropna())
    G_file = nx.DiGraph()
    for _, row in target_files.iterrows():
        sf = row['source_file']
        if pd.isna(sf):
            continue
        G_file.add_node(sf,
                        is_generated=bool(row.get('is_generated', False)),
                        compile_time_ms=safe_int(row.get('compile_time_ms')),
                        code_lines=safe_int(row.get('code_lines')),
                        git_churn=safe_int(row.get('git_churn')))
    for _, row in target_files.iterrows():
        source = row['source_file']
        if pd.isna(source):
            continue
        tree = parse_header_tree(row.get('header_tree'))
        stack = [(0, source)]
        for depth, header_path in tree:
            while len(stack) > 1 and stack[-1][0] >= depth:
                stack.pop()
            parent_path = stack[-1][1]
            if parent_path in source_paths and header_path in source_paths:
                if not G_file.has_edge(parent_path, header_path):
                    G_file.add_edge(parent_path, header_path, weight=0.0)
            stack.append((depth, header_path))
    # Weight edges by co-churn
    churns = nx.get_node_attributes(G_file, 'git_churn')
    max_churn = max(churns.values(), default=1) or 1
    for u, v in G_file.edges():
        G_file[u][v]['weight'] = min(churns.get(u, 0), churns.get(v, 0)) / max_churn
    return G_file


# ── Spectral 2-way partition ───────────────────────────────────────────────────
def contract_cpp_header_pairs(G_file):
    nodes = list(G_file.nodes())
    node_to_sn = {}
    stem_map = defaultdict(list)
    for path in nodes:
        p = Path(path)
        stem_map[str(p.parent / p.stem)].append(path)
    sn_members = {}
    cpp_exts = {'.cpp', '.cc', '.cxx', '.c'}
    hdr_exts = {'.h', '.hpp', '.hh', '.hxx'}
    for stem, paths in stem_map.items():
        cpp = [p for p in paths if Path(p).suffix in cpp_exts]
        hdr = [p for p in paths if Path(p).suffix in hdr_exts]
        if cpp and hdr:
            rep = cpp[0]
            for m in cpp + hdr:
                node_to_sn[m] = rep
            sn_members[rep] = cpp + hdr
        else:
            for p in paths:
                node_to_sn[p] = p
                sn_members[p] = [p]
    for path in nodes:
        if path not in node_to_sn:
            node_to_sn[path] = path
            sn_members[path] = [path]
    G_c = nx.Graph()
    for sn, members in sn_members.items():
        G_c.add_node(sn,
                     compile_time_ms=sum(G_file.nodes[m].get('compile_time_ms', 0) for m in members if m in G_file),
                     is_generated=any(G_file.nodes[m].get('is_generated', False) for m in members if m in G_file),
                     members=members)
    for u, v, data in G_file.edges(data=True):
        sn_u, sn_v = node_to_sn[u], node_to_sn[v]
        if sn_u == sn_v:
            continue
        if G_c.has_edge(sn_u, sn_v):
            G_c[sn_u][sn_v]['weight'] = G_c[sn_u][sn_v].get('weight', 0) + data.get('weight', 0)
        else:
            G_c.add_edge(sn_u, sn_v, weight=data.get('weight', 0))
    return G_c, node_to_sn


def pin_generated_files(G_c, heavy=1e4):
    for n in G_c.nodes():
        if G_c.nodes[n].get('is_generated', False):
            for nbr in list(G_c.neighbors(n)):
                G_c[n][nbr]['weight'] = G_c[n][nbr].get('weight', 0) + heavy


def spectral_2way(G_c):
    nodes = list(G_c.nodes())
    n = len(nodes)
    if n < 2:
        return {nd: 0 for nd in nodes}
    comps = list(nx.connected_components(G_c))
    G_work = G_c.subgraph(max(comps, key=len)).copy() if len(comps) > 1 else G_c
    work_nodes = list(G_work.nodes())
    nw = len(work_nodes)
    if nw < 2:
        return {nd: 0 for nd in nodes}
    idx = {nd: i for i, nd in enumerate(work_nodes)}
    W = nx.to_scipy_sparse_array(G_work, nodelist=work_nodes, weight='weight', format='csr')
    d = np.asarray(W.sum(axis=1)).flatten()
    D_inv_sqrt = csr_matrix((np.where(d > 0, 1.0 / np.sqrt(d), 0.0), (range(nw), range(nw))))
    L = csr_matrix(np.eye(nw)) - D_inv_sqrt @ W @ D_inv_sqrt
    k = min(3, nw - 1)
    vals, vecs = eigsh(L, k=k, which='SM')
    fiedler = vecs[:, 1] if k >= 2 else vecs[:, 0]
    partition = {}
    for nd in nodes:
        if nd in idx:
            partition[nd] = 0 if fiedler[idx[nd]] < 0 else 1
        else:
            partition[nd] = 0
    return partition


def metis_partition(G_c, nparts):
    nodes = list(G_c.nodes())
    if len(nodes) < nparts:
        return {nd: i % nparts for i, nd in enumerate(nodes)}
    idx = {nd: i for i, nd in enumerate(nodes)}
    adjacency = [[] for _ in nodes]
    for u, v, data in G_c.edges(data=True):
        adjacency[idx[u]].append(idx[v])
        adjacency[idx[v]].append(idx[u])
    try:
        _, membership = pymetis.part_graph(nparts, adjacency=adjacency)
        return {nodes[i]: membership[i] for i in range(len(nodes))}
    except Exception:
        return {nd: i % nparts for i, nd in enumerate(nodes)}


# ── Run split analysis for each candidate ─────────────────────────────────────
split_results = []

for target_name in sorted(split_candidates_set):
    G_file = build_intra_target_graph(target_name)
    if G_file.number_of_nodes() < 4:
        continue

    G_c, n2s = contract_cpp_header_pairs(G_file)
    pin_generated_files(G_c)

    # Spectral 2-way
    sp_partition = spectral_2way(G_c)
    sp_parts = defaultdict(list)
    for nd, part in sp_partition.items():
        sp_parts[part].extend(G_c.nodes[nd].get('members', [nd]))

    # METIS 2-way
    mt_partition = metis_partition(G_c, nparts=2)
    mt_parts = defaultdict(list)
    for nd, part in mt_partition.items():
        mt_parts[part].extend(G_c.nodes[nd].get('members', [nd]))

    # Cross-partition includes (spectral)
    cross_includes = sum(
        1 for u, v in G_file.edges()
        if sp_partition.get(n2s.get(u, u)) != sp_partition.get(n2s.get(v, v))
    )

    # Compile time per partition
    sp_times = defaultdict(int)
    for nd, part in sp_partition.items():
        sp_times[part] += G_c.nodes[nd].get('compile_time_ms', 0)
    times = list(sp_times.values())
    balance = min(times) / max(times) if max(times) > 0 else 1.0
    total_ms = safe_int(target_metrics_idx.loc[target_name].get('compile_time_sum_ms') if target_name in target_metrics_idx.index else 0)

    # Groups each partition falls into
    grp = group_map.get(target_name, 'unknown')

    # Simulate the split
    sp_file_groups = [list(sp_parts[k]) for k in sorted(sp_parts)]
    split_sim = simulate_split(G, target_name, sp_file_groups, target_df)

    gen_files_in_target = sum(1 for n in G_file.nodes() if G_file.nodes[n].get('is_generated', False))

    split_results.append({
        'cmake_target': target_name,
        'feature_group': grp,
        'file_count': G_file.number_of_nodes(),
        'compile_time_sum_ms': total_ms,
        'cross_partition_includes': cross_includes,
        'balance_ratio': round(balance, 3),
        'gen_files': gen_files_in_target,
        'recommended_k': 2,
        'method': 'spectral',
        'compile_saving_ms': int(total_ms * (1 - balance) * 0.3),  # conservative estimate
        'notes': '; '.join(split_sim['notes']),
    })

_SPLIT_COLS = ['cmake_target', 'feature_group', 'file_count', 'compile_time_sum_ms',
              'cross_partition_includes', 'balance_ratio', 'gen_files',
              'compile_saving_ms', 'recommended_k', 'method', 'notes']
if split_results:
    split_recs = pd.DataFrame(split_results).sort_values('compile_time_sum_ms', ascending=False)
    display(split_recs[[c for c in _SPLIT_COLS[:8] if c in split_recs.columns]].reset_index(drop=True))
else:
    split_recs = pd.DataFrame(columns=_SPLIT_COLS)
print(f'Split proposals: {len(split_recs)}')

# ── Persist split recommendations ─────────────────────────────────────────────
results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)
split_recs.to_parquet('../data/results/split_recommendations.parquet', index=False)
print('\nSaved: data/results/split_recommendations.parquet')


## 4. Interface Extraction Proposals

For thin cross-group dependencies: identify the specific headers used, propose a lightweight INTERFACE_LIBRARY wrapper target, and estimate compile time impact.


In [ ]:
# ── For thin deps: identify specific headers used ─────────────────────────────
def get_headers_used_from(depending_target, depended_target):
    """Return set of headers from depended_target that are included by depending_target."""
    dep_files = set(file_df[file_df['cmake_target'] == depended_target]['source_file'].dropna())
    src_files = file_df[file_df['cmake_target'] == depending_target]
    used = set()
    for _, row in src_files.iterrows():
        tree = parse_header_tree(row.get('header_tree'))
        for _, hdr in tree:
            if hdr in dep_files:
                used.add(hdr)
    return used


interface_proposals = []

# Consider thin cross-group edges
thin_cross = cross_df[cross_df['is_thin']].copy()

for _, row in thin_cross.head(20).iterrows():  # limit to top 20 for performance
    src_t = row['source_target']
    dst_t = row['dest_target']
    used_hdrs = get_headers_used_from(src_t, dst_t)
    if not used_hdrs:
        continue

    # Estimate compile time: fraction of dst target's compile time
    if dst_t in target_metrics_idx.index:
        dst_row = target_metrics_idx.loc[dst_t]
        dst_total_files = safe_int(dst_row.get('file_count'))
        dst_compile_ms = safe_int(dst_row.get('compile_time_sum_ms'))
        fraction = len(used_hdrs) / max(dst_total_files, 1)
        estimated_compile_ms = int(dst_compile_ms * fraction)
    else:
        fraction = 0.0
        estimated_compile_ms = 0

    # Proposed interface target name
    proposed_name = f'{dst_t}_iface'

    interface_proposals.append({
        'depending_target': src_t,
        'depended_target': dst_t,
        'source_group': row['source_group'],
        'dest_group': row['dest_group'],
        'used_headers': len(used_hdrs),
        'used_header_paths': '; '.join(sorted(used_hdrs)[:5]),
        'fraction_of_target': round(fraction, 3),
        'proposed_interface_target': proposed_name,
        'estimated_compile_ms': estimated_compile_ms,
        'forced_executables': row['forced_executables'],
    })

if interface_proposals:
    iface_df = pd.DataFrame(interface_proposals).sort_values('forced_executables', ascending=False)
else:
    iface_df = pd.DataFrame(interface_proposals)
    print('No iface df found (dataset too small).')

print(f'Interface extraction proposals: {len(iface_df)}')
if not iface_df.empty:
    display(iface_df[['depending_target', 'depended_target', 'dest_group',
                       'used_headers', 'fraction_of_target',
                       'proposed_interface_target', 'estimated_compile_ms',
                       'forced_executables']].reset_index(drop=True))

    # Bar chart of forced executables by proposal
    fig, ax = plt.subplots(figsize=(12, max(4, len(iface_df) * 0.45)))
    labels = [
        f"{r['depending_target'][:20]} → {r['depended_target'][:20]}"
        for _, r in iface_df.iterrows()
    ]
    ax.barh(range(len(iface_df)), iface_df['forced_executables'].values, color='steelblue')
    ax.set_yticks(range(len(iface_df)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Executables forced into extra feature group')
    ax.set_title('Interface Extraction Proposals by Impact')
    plt.tight_layout()
    plt.show()
else:
    print('No thin cross-group edges found. All cross-group dependencies appear to be thick.')




## 5. Target Merge Proposals

Within each feature group: identify targets with high mutual dependency, same owning team, and small size consumed by only one other target. Evaluate using `simulate_merge()`.


In [ ]:
# ── Find intra-group merge candidates ─────────────────────────────────────────
EXCLUDE_TYPES_MERGE = {'INTERFACE_LIBRARY', 'UTILITY'}
merge_proposals = []

for grp_name in feature_groups_only:
    grp_targets = set(fg_df[fg_df['feature_group'] == grp_name]['cmake_target'])
    grp_targets = {t for t in grp_targets if t in G}
    if len(grp_targets) < 2:
        continue

    # Find pairs with mutual edges within the group
    for t_a, t_b in itertools.combinations(sorted(grp_targets), 2):
        has_edge = G.has_edge(t_a, t_b) or G.has_edge(t_b, t_a)
        if not has_edge:
            continue

        # Same owning team?
        owner_a = ownership_map.get(t_a, 'unknown')
        owner_b = ownership_map.get(t_b, 'unknown')
        same_owner = (owner_a == owner_b) or (owner_a == 'unknown' or owner_b == 'unknown')

        # Small targets consumed by few things
        rev = G.reverse()
        dependants_a = set(G.predecessors(t_a)) | set(nx.descendants(rev, t_a))
        dependants_b = set(G.predecessors(t_b)) | set(nx.descendants(rev, t_b))
        total_dependants = len(dependants_a | dependants_b)

        # Skip very large or very large fan-out targets
        files_a = safe_int(target_metrics_idx.loc[t_a].get('authored_file_count') if t_a in target_metrics_idx.index else 0)
        files_b = safe_int(target_metrics_idx.loc[t_b].get('authored_file_count') if t_b in target_metrics_idx.index else 0)
        if files_a + files_b > 40:
            continue
        if total_dependants > 20:
            continue

        result = simulate_merge(G, [t_a, t_b], target_df)

        merge_proposals.append({
            'feature_group': grp_name,
            'target_a': t_a,
            'target_b': t_b,
            'owning_team': owner_a,
            'same_owner': same_owner,
            'files_total': files_a + files_b,
            'dependants': total_dependants,
            'savings_ms': result['savings_ms'],
            'before_ms': result['before_ms'],
            'notes': '; '.join(result['notes']),
        })

if merge_proposals:
    merge_df = pd.DataFrame(merge_proposals).sort_values('savings_ms', ascending=False)
else:
    merge_df = pd.DataFrame(merge_proposals)
    print('No merge df found (dataset too small).')

print(f'Merge proposals: {len(merge_df)}')
if not merge_df.empty:
    display(merge_df[['feature_group', 'target_a', 'target_b', 'owning_team',
                       'same_owner', 'files_total', 'dependants',
                       'savings_ms', 'before_ms']].head(20).reset_index(drop=True))

    fig, ax = plt.subplots(figsize=(10, max(4, min(len(merge_df), 15) * 0.45)))
    top_merges = merge_df[merge_df['savings_ms'] > 0].head(15)
    if not top_merges.empty:
        labels = [f"{r['target_a'][:18]} + {r['target_b'][:18]}" for _, r in top_merges.iterrows()]
        colors = ['#55A868' if r['same_owner'] else 'steelblue' for _, r in top_merges.iterrows()]
        ax.barh(range(len(top_merges)), top_merges['savings_ms'].values, color=colors)
        ax.set_yticks(range(len(top_merges)))
        ax.set_yticklabels(labels, fontsize=8)
        ax.invert_yaxis()
        ax.set_xlabel('Estimated build time saving (ms)')
        ax.set_title('Top Merge Proposals by Build Time Saving')
        ax.legend(handles=[Patch(color='#55A868', label='Same owner'),
                            Patch(color='steelblue', label='Different owner')],
                  loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No merge candidates found within feature group boundaries.')




## 6. Executable Feature Mapping

Compare the current feature group dependency count per executable against the proposed layout after proposed changes. Report reduction and save to CSV.


In [ ]:
# ── Current feature group deps per executable ─────────────────────────────────
exe_feature_rows = []
for exe, grps in exe_group_required.items():
    exe_feature_rows.append({
        'executable': exe,
        'feature_groups_required': len(grps),
        'feature_groups': '; '.join(sorted(grps)),
        'scenario': 'current',
    })

# ── Simulate proposed layout ──────────────────────────────────────────────────
# Apply interface extraction: for each iface proposal, the depending_target
# would now only need the interface's group (if it were in a different group)
proposed_group_map = dict(group_map)  # copy

# For interface extractions: proposed_interface_target would be in the depended group
# but the depending_target now links to the interface instead of the full target.
# Simulate by removing the cross-group edge from the dependency set.
thin_cross_set = set(
    zip(cross_df[cross_df['is_thin']]['source_target'],
        cross_df[cross_df['is_thin']]['dest_target'])
)

# Build proposed exe_group_required (with thin edges removed from group requirement)
proposed_exe_grps: dict[str, set[str]] = defaultdict(set)
for _, row in exe_lib_df.iterrows():
    lib = row['library']
    exe = row['executable']
    grp = proposed_group_map.get(lib)
    if not grp or grp == 'core':
        continue
    # Check if the path from exe to this lib goes through any thin edge we'd extract
    # Simplified: just check if the direct edge pair is thin
    if (exe, lib) in thin_cross_set:
        continue  # would be handled via interface library — no longer forces group
    proposed_exe_grps[exe].add(grp)

for exe, grps in proposed_exe_grps.items():
    exe_feature_rows.append({
        'executable': exe,
        'feature_groups_required': len(grps),
        'feature_groups': '; '.join(sorted(grps)),
        'scenario': 'proposed',
    })

mapping_df = pd.DataFrame(exe_feature_rows)

# ── Comparison ─────────────────────────────────────────────────────────────────
if not mapping_df.empty and 'feature_groups_required' in mapping_df.columns:
    pivot = mapping_df.pivot_table(
        index='executable',
        columns='scenario',
        values='feature_groups_required',
        aggfunc='first',
    ).reset_index().rename_axis(None, axis=1)
else:
    pivot = pd.DataFrame(columns=['executable'])
    print('No executable feature mapping data (dataset too small).')

if 'proposed' in pivot.columns and 'current' in pivot.columns:
    pivot['delta'] = pivot['proposed'] - pivot['current']
    pivot = pivot.sort_values('delta')
    print('Feature group reduction per executable:')
    display(pivot.reset_index(drop=True))
    total_reduction = -pivot['delta'].sum()
    print(f'\nTotal feature-group-dependency reductions: {total_reduction}')

# ── Save ──────────────────────────────────────────────────────────────────────
results_dir.mkdir(parents=True, exist_ok=True)
mapping_df.to_csv('../data/results/optimised_feature_mapping.csv', index=False)
print('Saved: data/results/optimised_feature_mapping.csv')

# ── Visualisation ─────────────────────────────────────────────────────────────
if not mapping_df.empty and 'scenario' in mapping_df.columns:

    for ax, scenario, color in zip(axes, ['current', 'proposed'], ['steelblue', '#55A868']):
        sub = mapping_df[mapping_df['scenario'] == scenario]
        if sub.empty:
            continue
        sub = sub.sort_values('feature_groups_required', ascending=False)
        ax.bar(range(len(sub)), sub['feature_groups_required'].values, color=color, alpha=0.8)
        ax.set_xticks(range(len(sub)))
        ax.set_xticklabels(sub['executable'].apply(lambda x: x[:20]).values,
                           rotation=45, ha='right', fontsize=8)
        ax.set_ylabel('Feature groups required')
        ax.set_title(f'{scenario.capitalize()}: Feature Groups per Executable')

    plt.tight_layout()
    plt.show()


## 7. Constrained Optimisation (Optional)

Use `simulated_annealing_partition()` to search for improved group assignments. Cost function weights: cross_group_edges, cross_group_compile_time, team_boundary_violations.


In [ ]:
# ── Simulated annealing on the feature group assignment ───────────────────────
# Start from the current Leiden-derived assignment; optimise for cross-group edges.

cost_weights = {
    'cross_group_edges': 1.0,
    'cross_group_compile_time': 0.001,  # scaled: ms → seconds
    'team_boundary_violations': 0.5,
}

# Only optimise non-core, non-utility targets that are in the graph
sa_targets = fg_df[
    (fg_df['feature_group'] != 'core') &
    (fg_df['cmake_target'].isin(set(G.nodes())))
].copy()

print(f'SA optimisation: {len(sa_targets)} non-core targets, '
      f'{len(sa_targets["feature_group"].unique())} groups')

if len(sa_targets) > 0 and len(sa_targets['feature_group'].unique()) >= 2:
    # Count initial cross-group edges
    initial_gm = dict(zip(fg_df['cmake_target'], fg_df['feature_group']))
    initial_cross = sum(
        1 for u, v in G.edges()
        if initial_gm.get(u, 'core') != initial_gm.get(v, 'core')
        and initial_gm.get(u, 'core') != 'core'
        and initial_gm.get(v, 'core') != 'core'
    )
    print(f'Initial cross-group edges: {initial_cross}')

    optimised_fg = simulated_annealing_partition(
        G,
        initial_assignment=sa_targets[['cmake_target', 'feature_group']],
        cost_weights=cost_weights,
        iterations=50_000,
        seed=42,
    )

    # Combine with core targets
    core_rows = fg_df[fg_df['feature_group'] == 'core'][['cmake_target', 'feature_group']]
    final_fg = pd.concat([core_rows, optimised_fg], ignore_index=True)

    # Count optimised cross-group edges
    opt_gm = dict(zip(final_fg['cmake_target'], final_fg['feature_group']))
    optimised_cross = sum(
        1 for u, v in G.edges()
        if opt_gm.get(u, 'core') != opt_gm.get(v, 'core')
        and opt_gm.get(u, 'core') != 'core'
        and opt_gm.get(v, 'core') != 'core'
    )
    print(f'Optimised cross-group edges: {optimised_cross}')
    print(f'Reduction: {initial_cross - optimised_cross} ({(initial_cross - optimised_cross) / max(initial_cross, 1):.1%})')

    # Compare group size changes
    initial_sizes = sa_targets.groupby('feature_group')['cmake_target'].count()
    opt_sizes = optimised_fg.groupby('feature_group')['cmake_target'].count()
    comparison = pd.DataFrame({'initial': initial_sizes, 'optimised': opt_sizes}).fillna(0).astype(int)
    comparison['delta'] = comparison['optimised'] - comparison['initial']
    print('\nGroup size changes:')
    display(comparison[comparison['delta'] != 0].reset_index())

    # Plot: initial vs optimised cross-group edges by group pair
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    for ax, gm, title in [
        (axes[0], initial_gm, f'Initial: {initial_cross} cross-group edges'),
        (axes[1], opt_gm, f'Optimised: {optimised_cross} cross-group edges'),
    ]:
        cross_counts = defaultdict(int)
        for u, v in G.edges():
            gu, gv = gm.get(u, 'core'), gm.get(v, 'core')
            if gu != gv and gu != 'core' and gv != 'core':
                cross_counts[(gu, gv)] += 1
        pairs = list(cross_counts.keys())
        if pairs:
            all_grps = sorted({g for pair in pairs for g in pair})
            heat = pd.DataFrame(0, index=all_grps, columns=all_grps)
            for (gu, gv), cnt in cross_counts.items():
                heat.loc[gu, gv] = cnt
            sns.heatmap(heat, annot=True, fmt='d', cmap='YlOrRd',
                        linewidths=0.5, ax=ax, cbar=False)
        ax.set_title(title)
    plt.tight_layout()
    plt.show()
else:
    print('Skipping SA optimisation: insufficient non-core targets or feature groups.')